In [ ]:
import os
import json
import scipy.interpolate
import numpy as np
import torch
import torchaudio
import torchaudio.functional as F
import matplotlib.pyplot as plt
from IPython.display import Audio, display
from torchaudio.transforms import Resample, MelSpectrogram, AmplitudeToDB

In [ ]:
#url = "https://download.pytorch.org/torchaudio/tutorial-assets/Lab41-SRI-VOiCES-src-sp0307-ch127535-sg0042.wav"
video_id = "QID80I1IRyI"

path_wsl = f"/mnt/c/Users/dexter.neo/Desktop/LLMVS/{video_id}.mp4"
path_linux = f"/home/dexter/LLaVA-VLS/{video_id}.mp4"

# Check which one actually exists
if os.path.exists(path_wsl):
    url = path_wsl
    print(f"Found file in WSL path: {url}")
elif os.path.exists(path_linux):
    url = path_linux
    print(f"Found file in Linux path: {url}")
    

waveform, sample_rate = torchaudio.load(url)
waveform_mono = waveform[0]
duration_sec = waveform.shape[1] / sample_rate

print(f"Waveform shape: {waveform.shape}")
print(f"Sample rate: {sample_rate} Hz")

In [ ]:
low_cut = 6000
high_cut = 7000

wave_temp = F.highpass_biquad(waveform_mono, sample_rate, cutoff_freq=low_cut)
waveform_mono = F.lowpass_biquad(wave_temp, sample_rate, cutoff_freq=high_cut)

In [ ]:
#display(Audio(waveform_mono, rate=sample_rate))

In [ ]:
#display(Audio(waveform[0], rate=sample_rate))

In [ ]:
#waveform = waveform.mean(0)
sample_rate = 16000
n_fft = 512

spec_transform = torchaudio.transforms.Spectrogram(
    n_fft=n_fft, 
    win_length=None, 
    hop_length=sample_rate,
    power=2.0
)

# Generate spectrogram and convert to Decibels (Log Scale) for visualization
spectrogram = spec_transform(waveform_mono)
spectrogram_db = torchaudio.transforms.AmplitudeToDB()(spectrogram)

# 2. Configure Transforms (1 sample per second alignment)
hop_length = sample_rate 
n_mels = 128 # Number of Mel bands

# --- Mel Spectrogram ---
mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=sample_rate,
    n_fft=n_fft,
    hop_length=sample_rate,
    n_mels=n_mels,
    center=False,
    power=2.0
)
mel_spectrogram_db = torchaudio.transforms.AmplitudeToDB()(mel_transform(waveform_mono))

In [ ]:
# 3. Visualization with Shared Axes
# sharex=True locks the x-axis of both plots together
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

# --- Plot 1: Waveform ---
time_axis = torch.linspace(0, duration_sec, steps=len(waveform.mean(0)))
ax1.plot(time_axis, waveform_mono.numpy(), linewidth=0.5)
ax1.set_title("Waveform")
ax1.set_ylabel("Amplitude")
ax1.grid(True, alpha=0.3)
ax1.tick_params(labelbottom=True)

# --- Plot 2: Spectrogram ---
# extent=[left, right, bottom, top] maps the array pixels to physical units
img = ax2.imshow(spectrogram_db.numpy(), 
           aspect='auto', 
           origin='lower', 
           cmap='inferno',
           extent=[0, duration_sec, 0, sample_rate // 2])

ax2.set_title("Spectrogram")
ax2.set_ylabel("Frequency (Hz)")
ax2.set_xlabel("Time (seconds)")

cbar = fig.colorbar(img, ax=ax2, format='%+2.0f dB')
cbar.set_label('Intensity (Decibels)')

img3 = ax3.imshow(mel_spectrogram_db.numpy(), 
                 aspect='auto', 
                 origin='lower', 
                 cmap='inferno',
                 extent=[0, duration_sec, 0, n_mels]) # Y-axis is Mel bins (0-128)
ax3.set_title("Mel Spectrogram")
ax3.set_ylabel("Mel Filter Banks")
ax3.set_xlabel("Time (seconds)")
fig.colorbar(img3, ax=ax3, format='%+2.0f dB', label='Intensity (dB)')

# Fine-tuning alignment
plt.xlim(0, duration_sec) # Enforce strict start/end for both
plt.tight_layout()
plt.show()

In [ ]:
try:
    with open(f"/home/dexter/LLaVA-VLS/{video_id}_heatmap.json", 'r') as f:
        data = json.load(f)
except FileNotFoundError:
    with open(f"/mnt/c/Users/dexter.neo/Desktop/LLMVS/{video_id}.mp4", 'r') as f:
        data = json.load(f)

# 3. Prepare Data for Step Plot
times = [entry["start_time"] for entry in data] + [data[-1]["end_time"]]
values = [entry["value"] for entry in data] + [data[-1]["value"]]
plt.plot(times, values)

In [ ]:
target_time_axis = np.linspace(0, times[-1], num=waveform.shape[1])

# 3. Create Interpolation Function (Step Function)
# kind='previous' ensures that for the range [0.0 to 13.9], the value remains 0.181...
# This creates a "staircase" function rather than a smooth slope.
f = scipy.interpolate.interp1d(times, values, kind='previous', fill_value="extrapolate")

# 4. Resample
# Now we calculate the 'excitement score' for every single sample in the audio
excitement_per_sample = f(target_time_axis)

# Convert to tensor for PyTorch compatibility
excitement_tensor = torch.tensor(excitement_per_sample).float()

# --- Verification ---
print(f"Original bins: {len(values)}")
print(f"Upsampled shape: {excitement_tensor.shape}")

In [ ]:
plt.figure(figsize=(12, 6))
plt.title("Value (Importance)")

# Plot Waveform (Normalized for visualization)
# We normalize waveform to 0-1 range just to plot it behind the mask clearly
wave_norm = waveform_mono.numpy()
wave_norm = (wave_norm - wave_norm.min()) / (wave_norm.max() - wave_norm.min())
plt.plot(target_time_axis, wave_norm, color='gray', alpha=0.3, label="Audio")

# Plot Excitement Mask
plt.plot(target_time_axis, excitement_tensor.numpy(), color='blue', label="Excitement Mask")

plt.xlabel("Time (seconds)")
plt.legend(loc="upper right")
plt.show()

In [ ]:
# Target: Match the Spectrogram's width (number of seconds)
num_frames = mel_spectrogram_db.shape[1]
spec_time_axis = np.linspace(0, duration_sec, num=num_frames)

# Resample Excitement to match Spectrogram frames (Step Function)
f = scipy.interpolate.interp1d(times, values, kind='previous', fill_value="extrapolate")
excitement_aligned = f(spec_time_axis)

audio_energy = mel_spectrogram_db.mean(dim=0).numpy()
audio_energy_norm = (audio_energy - audio_energy.min()) / (audio_energy.max() - audio_energy.min())

threshold_val = 0.9

# --- 4. Visualization ---
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

# Subplot 1: Mel Spectrogram
img = ax1.imshow(mel_spectrogram_db.numpy(), aspect='auto', origin='lower', cmap='inferno',
           extent=[0, duration_sec, 0, n_mels])
ax1.set_title("Visual: Mel Spectrogram (Frequency Content)")
ax1.set_ylabel("Mel Bins")
fig.colorbar(img, ax=ax1, format='%+2.0f dB')

# Subplot 2: Correlation Check
ax2.set_title("Correlation Check: Excitement Score vs Audio Energy")

# Plot Excitement (Blue)
ax2.plot(spec_time_axis, excitement_aligned, color='cyan', linewidth=2, label="Your Excitement Score")
ax2.fill_between(spec_time_axis, excitement_aligned, color='cyan', alpha=0.1)
ax2.plot(spec_time_axis, audio_energy_norm, color='orange', linestyle='--', linewidth=1.5, label="Spectral Energy (Normalized)")
ax2.axhline(threshold_val, color='red', linestyle='--', label='Threshold')

ax2.set_ylabel("Normalized Score")
ax2.set_xlabel("Time (seconds)")
ax2.legend(loc="upper right")
ax2.grid(True, alpha=0.3)


wave_norm = waveform_mono.numpy()
wave_norm = (wave_norm - wave_norm.min()) / (wave_norm.max() - wave_norm.min())
ax3.plot(target_time_axis, wave_norm, color='gray', alpha=0.3, label="Audio")
ax3.plot(target_time_axis, excitement_tensor.numpy(), color='blue', label="Excitement Mask")
ax3.axhline(threshold_val, color='red', linestyle='--', label='Threshold')


plt.xlim(0, duration_sec)
plt.tight_layout()
plt.show()

In [ ]:
# --- 2. Calculate "Action Score" (The High-Freq Max-Pool Method) ---
# We use a small window (10ms) to catch sharp impacts
n_fft_action = 512
hop_length_action = 256 

transform_action = torchaudio.transforms.Spectrogram(
    n_fft=n_fft_action,
    hop_length=hop_length_action,
    power=2.0
)
spec_action = transform_action(waveform_mono) # Shape: [257, time_steps]

# Define "Action" Frequencies
# With n_fft=512, each bin is ~86Hz. 
# Bin 32 is approx 2.75 kHz. 
# We sum energy from 2.75 kHz up to Nyquist (22kHz) to capture "clashing/hissing"
high_freq_start_bin = 32 
high_freq_energy = spec_action[high_freq_start_bin:, :].sum(dim=0)

# "Max Pool" into 1-second buckets
# 1. Calculate how many frames exist in 1 second
frames_per_sec = sample_rate // hop_length_action

# 2. Reshape into [seconds, frames_per_sec]
num_seconds = int(high_freq_energy.shape[0] / frames_per_sec)
total_frames_needed = num_seconds * frames_per_sec

# Truncate slightly to fit perfect grid
energy_matrix = high_freq_energy[:total_frames_needed].view(num_seconds, frames_per_sec)

# 3. Take the MAX value in each second (This captures the "spike")
action_score = energy_matrix.max(dim=1).values

# Normalize to 0-1 for plotting
action_score_norm = (action_score - action_score.min()) / (action_score.max() - action_score.min())
time_axis_action = np.arange(num_seconds)


# --- 4. Plot Comparison ---
plt.figure(figsize=(14, 8))

# Plot 1: The New "Action" Detector
plt.subplot(2, 1, 1)
plt.title("High-Frequency Action Detector (Max-Pooled)")
plt.plot(time_axis_action, action_score_norm.numpy(), color='red', linewidth=1.5, label="High-Freq Transient Score")
plt.fill_between(time_axis_action, action_score_norm.numpy(), color='red', alpha=0.1)
plt.ylabel("Intensity (Normalized)")
plt.grid(True, alpha=0.3)
plt.legend(loc="upper right")
plt.xlim(0, num_seconds)

# Plot 2: Overlay
plt.subplot(2, 1, 2)
plt.title("Overlay: Action vs. Original Excitement")
plt.plot(time_axis_action, action_score_norm.numpy(), color='red', linewidth=1, alpha=0.6, label="Action (Sharp)")
plt.plot(time_axis_action, excitement_aligned, color='cyan', linewidth=2, label="Original Excitement (Blocky)")
plt.ylabel("Normalized Score")
plt.xlabel("Time (seconds)")
plt.grid(True, alpha=0.3)
plt.legend(loc="upper right")
plt.xlim(0, num_seconds)

plt.tight_layout()
plt.show()

In [ ]:
from transformers import ASTFeatureExtractor, ASTForAudioClassification

# 1. Load Model and Processor
model_id = "MIT/ast-finetuned-audioset-10-10-0.4593"
feature_extractor = ASTFeatureExtractor.from_pretrained(model_id)
model = ASTForAudioClassification.from_pretrained(model_id)

# 2. Define target labels
TARGET_LABELS = ["Shout", "Yell", "Scream", "Fighting", "Explosion"]

def detect_events(waveform_tensor, chunk_duration_ms=1000, overlap=0.5):
    # Ensure input is a numpy array (move to CPU if needed)
    if torch.is_tensor(waveform_tensor):
        audio_data = waveform_tensor.cpu().numpy()
    else:
        audio_data = waveform_tensor

    sr = 16000
    
    # Calculate window size in SAMPLES, not ms
    # 1000ms @ 16k Hz = 16000 samples
    chunk_samples = int((chunk_duration_ms / 1000) * sr)
    step_samples = int(chunk_samples * (1 - overlap))
    
    events = []
    
    # Sliding window loop (iterating by samples)
    for start_sample in range(0, len(audio_data) - chunk_samples, step_samples):
        # Slice the numpy array directly
        chunk = audio_data[start_sample : start_sample + chunk_samples]
        
        # Inference
        inputs = feature_extractor(chunk, sampling_rate=sr, return_tensors="pt")
        
        with torch.no_grad():
            outputs = model(**inputs)
            
        # Get top prediction
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
        score, label_idx = torch.max(probs, dim=-1)
        label = model.config.id2label[label_idx.item()]
        
        timestamp_sec = start_sample / sr
        # if score > 0.7:
        print(f"Detected {label} with score {score} at {timestamp_sec:.2f}s")
            
    return events

# Usage: Pass the tensor directly
detect_events(waveform_mono)

In [ ]:
from transformers import ASTFeatureExtractor, ASTForAudioClassification

# 1. Load Model and Processor
model_id = "MIT/ast-finetuned-audioset-10-10-0.4593"
feature_extractor = ASTFeatureExtractor.from_pretrained(model_id)
model = ASTForAudioClassification.from_pretrained(model_id)

# 2. Define target labels
TARGET_LABELS = ["Shout", "Yell", "Scream", "Fighting", "Explosion"]

def detect_events(waveform_tensor, chunk_duration_ms=1000, overlap=0.5):
    # Ensure input is a numpy array (move to CPU if needed)
    if torch.is_tensor(waveform_tensor):
        audio_data = waveform_tensor.cpu().numpy()
    else:
        audio_data = waveform_tensor

    sr = 16000
    
    # Calculate window size in SAMPLES, not ms
    # 1000ms @ 16k Hz = 16000 samples
    chunk_samples = int((chunk_duration_ms / 1000) * sr)
    step_samples = int(chunk_samples * (1 - overlap))
    
    events = []
    
    # Sliding window loop (iterating by samples)
    for start_sample in range(0, len(audio_data) - chunk_samples, step_samples):
        # Slice the numpy array directly
        chunk = audio_data[start_sample : start_sample + chunk_samples]
        
        # Inference
        inputs = feature_extractor(chunk, sampling_rate=sr, return_tensors="pt")
        
        with torch.no_grad():
            outputs = model(**inputs)
            
        # Get top prediction
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
        score, label_idx = torch.max(probs, dim=-1)
        label = model.config.id2label[label_idx.item()]
        
        timestamp_sec = start_sample / sr
        # if score > 0.7:
        print(f"Detected {label} with score {score} at {timestamp_sec:.2f}s")
            
    return events

# Usage: Pass the tensor directly
detect_events(waveform_mono)